<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/research-v2/notebooks/20_v2_lossless_speculative_decoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 20: Lossless Self-Speculative Decoding

This notebook implements a complete lossless self-speculative decoding
prototype using DistilGPT-2 as the target model and learned latent
transition models as drafters.

The experiment compares:

1. Vanilla autoregressive decoding
2. Multi-step latent drafting
3. Decoder-KL latent drafting
4. Local-LK latent drafting

Draft horizons:

$$
\gamma \in \{2,4,8\}.
$$

The decoder implements:

- latent draft generation;
- parallel target verification;
- standard speculative acceptance;
- rejection correction;
- bonus-token sampling when all drafts are accepted;
- target KV-cache maintenance;
- synchronized wall-clock measurement.

Primary metrics:

- accepted draft tokens per verification round;
- final tokens per target verification;
- draft latency;
- verification latency;
- correction/update latency;
- total decoding latency;
- output tokens per second;
- speedup over vanilla autoregressive decoding.

The method is lossless because the final output distribution is
identical to that of the target model under standard speculative
sampling.

In [1]:
!pip -q install "transformers==4.57.1" pandas matplotlib seaborn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 177.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 72.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
!pip -q uninstall -y gradio gradio_client
!pip -q install "transformers==4.57.1" "huggingface-hub==0.36.2" pandas matplotlib seaborn scipy

In [1]:
import torch
import transformers
import huggingface_hub
import pandas
import scipy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Hugging Face Hub: 0.36.2
GPU available: True


In [2]:
import os
import gc
import json
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ma2288_nextlat"
)

PROMPT_SPLIT_PATH = (
    PROJECT_DIR
    / "data_v3"
    / "notebook19_prompt_splits_seed42.pt"
)

CHECKPOINT_PATHS = {
    "Multi-step": (
        PROJECT_DIR
        / "checkpoints"
        / "multistep_transition_seed42.pt"
    ),

    "Decoder-KL": (
        PROJECT_DIR
        / "checkpoints_v3"
        / "decoder_kl_c2_selected_seed42.pt"
    ),

    "Local-LK": (
        PROJECT_DIR
        / "checkpoints_v3"
        / "local_lk_c2_selected_seed42.pt"
    ),
}

OUTPUT_TABLE_DIR = (
    PROJECT_DIR
    / "results_v2"
    / "tables"
)

OUTPUT_FIGURE_DIR = (
    PROJECT_DIR
    / "results_v2"
    / "figures"
)

OUTPUT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_DIR.exists()
assert PROMPT_SPLIT_PATH.exists()

for method, path in CHECKPOINT_PATHS.items():
    assert path.exists(), (
        f"Missing {method}: {path}"
    )

print("All required files exist.")

All required files exist.


In [5]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

target_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

target_model.eval()

for parameter in target_model.parameters():
    parameter.requires_grad = False

token_embedding = target_model.get_input_embeddings()

HIDDEN_SIZE = target_model.config.n_embd
VOCAB_SIZE = target_model.config.vocab_size

print("Model:", MODEL_NAME)
print("Hidden size:", HIDDEN_SIZE)
print("Vocabulary size:", VOCAB_SIZE)
print("EOS token ID:", tokenizer.eos_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: distilgpt2
Hidden size: 768
Vocabulary size: 50257
EOS token ID: 50256


In [6]:
class ResidualLatentTransition(nn.Module):
    def __init__(
        self,
        hidden_size=768,
        token_embedding_size=768,
        intermediate_size=512,
    ):
        super().__init__()

        input_size = (
            hidden_size + token_embedding_size
        )

        self.input_normalization = nn.LayerNorm(
            input_size
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                intermediate_size,
            ),
            nn.GELU(),
            nn.Linear(
                intermediate_size,
                hidden_size,
            ),
        )

    def forward(
        self,
        hidden_state,
        token_embedding_value,
    ):
        model_input = torch.cat(
            [
                hidden_state,
                token_embedding_value,
            ],
            dim=-1,
        )

        normalized_input = self.input_normalization(
            model_input
        )

        delta = self.network(
            normalized_input
        )

        return hidden_state + delta

In [7]:
draft_models = {}

for method_name, checkpoint_path in (
    CHECKPOINT_PATHS.items()
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    assert "model_state_dict" in checkpoint

    draft_model = ResidualLatentTransition(
        hidden_size=HIDDEN_SIZE,
        token_embedding_size=HIDDEN_SIZE,
        intermediate_size=512,
    ).to(device)

    load_result = draft_model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    assert len(load_result.missing_keys) == 0
    assert len(load_result.unexpected_keys) == 0

    draft_model.eval()

    for parameter in draft_model.parameters():
        parameter.requires_grad = False

    draft_models[method_name] = draft_model

    print(
        f"{method_name}: "
        f"{checkpoint_path.name}"
    )

Multi-step: multistep_transition_seed42.pt
Decoder-KL: decoder_kl_c2_selected_seed42.pt
Local-LK: local_lk_c2_selected_seed42.pt


In [8]:
prompt_split_data = torch.load(
    PROMPT_SPLIT_PATH,
    map_location="cpu",
    weights_only=False,
)

test_prompt_tokens = prompt_split_data[
    "test_prompt_tokens"
].long()

assert test_prompt_tokens.shape == (
    80,
    32,
)

print("Test prompts:", test_prompt_tokens.shape)

for index in range(2):
    print(f"\nPrompt {index}:")
    print(
        repr(
            tokenizer.decode(
                test_prompt_tokens[
                    index
                ].tolist()
            )
        )
    )

Test prompts: torch.Size([80, 32])

Prompt 0:
" .<|endoftext|>His father died around 740 . Du Fu would have been allowed to enter the civil service because of his father 's rank , but he is thought"

Prompt 1:
' in domestic affairs .<|endoftext|>In the autumn of 744 , he met Li Bai ( Li Po ) for the first time , and the two poets formed a friendship'


In [9]:
def synchronize_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


class CudaTimer:
    def __enter__(self):
        synchronize_device()
        self.start_time = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        synchronize_device()
        self.elapsed_seconds = (
            time.perf_counter()
            - self.start_time
        )

In [10]:
sample_prompt = (
    test_prompt_tokens[:1].to(device)
)

with torch.inference_mode():
    sample_output = target_model(
        input_ids=sample_prompt,
        use_cache=True,
        output_hidden_states=True,
        return_dict=True,
    )

sample_cache = sample_output.past_key_values

print("Cache type:", type(sample_cache))
print(
    "Number of transformer layers:",
    len(sample_cache),
)

first_layer = sample_cache[0]

print(
    "First-layer object type:",
    type(first_layer),
)

if isinstance(first_layer, tuple):
    print(
        "First key shape:",
        first_layer[0].shape,
    )

    print(
        "First value shape:",
        first_layer[1].shape,
    )
else:
    print(
        "First-layer contents:",
        first_layer,
    )

Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of transformer layers: 6
First-layer object type: <class 'tuple'>
First key shape: torch.Size([1, 12, 32, 64])
First value shape: torch.Size([1, 12, 32, 64])


In [11]:
def get_cache_sequence_length(cache):
    if hasattr(cache, "get_seq_length"):
        return int(cache.get_seq_length())

    first_layer = cache[0]

    if isinstance(first_layer, tuple):
        return int(first_layer[0].shape[-2])

    raise TypeError(
        f"Unsupported cache type: {type(cache)}"
    )


def crop_cache(cache, maximum_length):
    if hasattr(cache, "crop"):
        cache.crop(maximum_length)
        return cache

    cropped_layers = []

    for layer in cache:
        key, value = layer

        cropped_key = key[
            ...,
            :maximum_length,
            :,
        ].contiguous()

        cropped_value = value[
            ...,
            :maximum_length,
            :,
        ].contiguous()

        cropped_layers.append(
            (
                cropped_key,
                cropped_value,
            )
        )

    return tuple(cropped_layers)

In [12]:
@torch.inference_mode()
def prefill_target(prompt_token_ids):
    prompt_token_ids = prompt_token_ids.to(
        device=device,
        dtype=torch.long,
    )

    attention_mask = torch.ones_like(
        prompt_token_ids
    )

    output = target_model(
        input_ids=prompt_token_ids,
        attention_mask=attention_mask,
        use_cache=True,
        output_hidden_states=True,
        return_dict=True,
    )

    last_hidden = (
        output.hidden_states[-1][
            :,
            -1,
            :,
        ].float()
    )

    next_token_logits = (
        output.logits[
            :,
            -1,
            :,
        ].float()
    )

    return {
        "cache": output.past_key_values,
        "last_hidden": last_hidden,
        "next_token_logits":
            next_token_logits,
        "sequence_length":
            prompt_token_ids.shape[1],
    }

In [13]:
@torch.inference_mode()
def advance_target_one_token(
    token_ids,
    cache,
    previous_sequence_length,
):
    token_ids = token_ids.to(
        device=device,
        dtype=torch.long,
    ).view(-1, 1)

    new_sequence_length = (
        previous_sequence_length + 1
    )

    attention_mask = torch.ones(
        (
            token_ids.shape[0],
            new_sequence_length,
        ),
        dtype=torch.long,
        device=device,
    )

    output = target_model(
        input_ids=token_ids,
        attention_mask=attention_mask,
        past_key_values=cache,
        use_cache=True,
        output_hidden_states=True,
        return_dict=True,
    )

    last_hidden = (
        output.hidden_states[-1][
            :,
            -1,
            :,
        ].float()
    )

    next_token_logits = (
        output.logits[
            :,
            -1,
            :,
        ].float()
    )

    return {
        "cache": output.past_key_values,
        "last_hidden": last_hidden,
        "next_token_logits":
            next_token_logits,
        "sequence_length":
            new_sequence_length,
    }

In [15]:
test_prompt = (
    test_prompt_tokens[:1].to(device)
)

prefill_state = prefill_target(
    test_prompt
)

first_next_token = (
    prefill_state[
        "next_token_logits"
    ].argmax(dim=-1)
)

cached_state = advance_target_one_token(
    token_ids=first_next_token,
    cache=prefill_state["cache"],
    previous_sequence_length=
        prefill_state[
            "sequence_length"
        ],
)

extended_context = torch.cat(
    [
        test_prompt,
        first_next_token.view(1, 1),
    ],
    dim=1,
)

with torch.inference_mode():
    full_output = target_model(
        input_ids=extended_context,
        use_cache=False,
        output_hidden_states=True,
        return_dict=True,
    )

full_last_hidden = (
    full_output.hidden_states[-1][
        :,
        -1,
        :,
    ].float()
)

full_next_logits = (
    full_output.logits[
        :,
        -1,
        :,
    ].float()
)

hidden_difference = (
    cached_state["last_hidden"]
    - full_last_hidden
).abs().max().item()

logit_difference = (
    cached_state["next_token_logits"]
    - full_next_logits
).abs().max().item()

cached_probability = F.softmax(
    cached_state["next_token_logits"],
    dim=-1,
)

full_probability = F.softmax(
    full_next_logits,
    dim=-1,
)

probability_difference = (
    cached_probability
    - full_probability
).abs().max().item()

cached_top1 = (
    cached_state["next_token_logits"]
    .argmax(dim=-1)
)

full_top1 = (
    full_next_logits
    .argmax(dim=-1)
)

top1_agreement = bool(
    torch.equal(
        cached_top1,
        full_top1,
    )
)

cache_sequence_length = (
    get_cache_sequence_length(
        cached_state["cache"]
    )
)

print(
    "Cached/full hidden difference:",
    hidden_difference,
)

print(
    "Cached/full logit difference:",
    logit_difference,
)

print(
    "Cached/full probability difference:",
    probability_difference,
)

print(
    "Cached top-1 token:",
    int(cached_top1.item()),
    repr(
        tokenizer.decode(
            cached_top1.tolist()
        )
    ),
)

print(
    "Full top-1 token:",
    int(full_top1.item()),
    repr(
        tokenizer.decode(
            full_top1.tolist()
        )
    ),
)

print(
    "Top-1 agreement:",
    top1_agreement,
)

print(
    "Cache sequence length:",
    cache_sequence_length,
)

NUMERICAL_ALIGNMENT_TOLERANCE = 5e-4

assert (
    hidden_difference
    < NUMERICAL_ALIGNMENT_TOLERANCE
)

assert (
    logit_difference
    < NUMERICAL_ALIGNMENT_TOLERANCE
)

assert probability_difference < 1e-5
assert top1_agreement

assert cache_sequence_length == (
    extended_context.shape[1]
)

print("KV-cache alignment passed.")

Cached/full hidden difference: 0.000118255615234375
Cached/full logit difference: 0.000110626220703125
Cached/full probability difference: 8.940696716308594e-07
Cached top-1 token: 423 ' have'
Full top-1 token: 423 ' have'
Top-1 agreement: True
Cache sequence length: 33
KV-cache alignment passed.


In [16]:
original_length = get_cache_sequence_length(
    cached_state["cache"]
)

cropped_cache = crop_cache(
    copy.deepcopy(
        cached_state["cache"]
    ),
    original_length - 1,
)

cropped_length = get_cache_sequence_length(
    cropped_cache
)

print("Original cache length:", original_length)
print("Cropped cache length:", cropped_length)

assert cropped_length == (
    original_length - 1
)

print("KV-cache crop passed.")

Original cache length: 33
Cropped cache length: 32
KV-cache crop passed.


In [17]:
print("=" * 90)
print("NOTEBOOK 20 — PHASE A SUMMARY")
print("=" * 90)

print("PyTorch:", torch.__version__)
print(
    "Transformers:",
    transformers.__version__,
)

print("Device:", device)
print("Target model:", MODEL_NAME)
print("Test prompts:", tuple(
    test_prompt_tokens.shape
))

print("\nDraft models:")

for method_name in draft_models:
    print("-", method_name)

print(
    "\nCached/full hidden difference:",
    hidden_difference,
)

print(
    "Cached/full logit difference:",
    logit_difference,
)

print(
    "Cache length after crop:",
    cropped_length,
)

print("\nPHASE A COMPLETE")

NOTEBOOK 20 — PHASE A SUMMARY
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Device: cuda
Target model: distilgpt2
Test prompts: (80, 32)

Draft models:
- Multi-step
- Decoder-KL
- Local-LK

Cached/full hidden difference: 0.000118255615234375
Cached/full logit difference: 0.000110626220703125
Cache length after crop: 32

PHASE A COMPLETE


In [18]:
def sample_from_probability(
    probability,
    generator,
):
    return torch.multinomial(
        probability,
        num_samples=1,
        replacement=True,
        generator=generator,
    ).squeeze(1)


def sample_from_logits(
    logits,
    generator,
    temperature=1.0,
):
    if temperature <= 0:
        return logits.argmax(
            dim=-1
        )

    probability = F.softmax(
        logits / temperature,
        dim=-1,
    )

    return sample_from_probability(
        probability,
        generator,
    )

In [19]:
@torch.inference_mode()
def generate_latent_draft_block(
    draft_model,
    initial_hidden,
    horizon,
    generator,
    temperature=1.0,
):
    current_hidden = (
        initial_hidden.clone()
    )

    draft_token_list = []
    draft_probability_list = []
    draft_logits_list = []

    for step in range(horizon):
        draft_logits = target_model.lm_head(
            current_hidden
        ).float()

        if temperature <= 0:
            draft_probability = F.softmax(
                draft_logits,
                dim=-1,
            )

            draft_token = (
                draft_logits.argmax(
                    dim=-1
                )
            )
        else:
            draft_probability = F.softmax(
                draft_logits / temperature,
                dim=-1,
            )

            draft_token = (
                sample_from_probability(
                    draft_probability,
                    generator,
                )
            )

        draft_token_list.append(
            draft_token
        )

        draft_probability_list.append(
            draft_probability
        )

        draft_logits_list.append(
            draft_logits
        )

        if step == horizon - 1:
            continue

        token_embeddings = token_embedding(
            draft_token
        ).float()

        current_hidden = draft_model(
            current_hidden,
            token_embeddings,
        )

    draft_token_ids = torch.stack(
        draft_token_list,
        dim=1,
    )

    return {
        "draft_token_ids":
            draft_token_ids,

        "draft_probabilities":
            draft_probability_list,

        "draft_logits":
            draft_logits_list,
    }

In [20]:
@torch.inference_mode()
def verify_target_block(
    draft_token_ids,
    cache,
    previous_sequence_length,
):
    horizon = draft_token_ids.shape[1]

    new_sequence_length = (
        previous_sequence_length
        + horizon
    )

    attention_mask = torch.ones(
        (
            draft_token_ids.shape[0],
            new_sequence_length,
        ),
        dtype=torch.long,
        device=device,
    )

    output = target_model(
        input_ids=draft_token_ids,
        attention_mask=attention_mask,
        past_key_values=cache,
        use_cache=True,
        output_hidden_states=True,
        return_dict=True,
    )

    return {
        "cache":
            output.past_key_values,

        # Logits at draft position i predict
        # the token after draft token i.
        "logits":
            output.logits.float(),

        "hidden_states":
            output.hidden_states[-1].float(),

        "sequence_length":
            new_sequence_length,
    }

In [21]:
@torch.inference_mode()
def speculative_decode_round(
    draft_model,
    target_state,
    horizon,
    sampling_generator,
    acceptance_generator,
    temperature=1.0,
    epsilon=1e-8,
):
    base_sequence_length = (
        target_state["sequence_length"]
    )

    draft_result = (
        generate_latent_draft_block(
            draft_model=draft_model,
            initial_hidden=target_state[
                "last_hidden"
            ],
            horizon=horizon,
            generator=sampling_generator,
            temperature=temperature,
        )
    )

    draft_token_ids = draft_result[
        "draft_token_ids"
    ]

    draft_probabilities = draft_result[
        "draft_probabilities"
    ]

    verification_result = (
        verify_target_block(
            draft_token_ids=
                draft_token_ids,
            cache=target_state["cache"],
            previous_sequence_length=
                base_sequence_length,
        )
    )

    # p_1 comes from the final prompt/committed token.
    # p_2 ... p_K come from verifier logits after
    # draft tokens 1 ... K-1.
    target_logits_for_drafts = [
        target_state[
            "next_token_logits"
        ]
    ]

    for step in range(
        1,
        horizon,
    ):
        target_logits_for_drafts.append(
            verification_result[
                "logits"
            ][:, step - 1, :]
        )

    acceptance_probabilities = []
    accepted_count = 0
    rejected_position = None
    correction_token = None
    bonus_token = None

    for step in range(horizon):
        target_logits = (
            target_logits_for_drafts[
                step
            ]
        )

        draft_probability = (
            draft_probabilities[step]
        )

        current_draft_token = (
            draft_token_ids[:, step]
        )

        if temperature <= 0:
            target_token = (
                target_logits.argmax(
                    dim=-1
                )
            )

            accepted = bool(
                torch.equal(
                    current_draft_token,
                    target_token,
                )
            )

            acceptance_probability = (
                torch.ones(
                    current_draft_token.shape[0],
                    device=device,
                )
                if accepted
                else torch.zeros(
                    current_draft_token.shape[0],
                    device=device,
                )
            )
        else:
            target_probability = F.softmax(
                target_logits / temperature,
                dim=-1,
            )

            selected_target_probability = (
                target_probability
                .gather(
                    1,
                    current_draft_token
                    .unsqueeze(1),
                )
                .squeeze(1)
            )

            selected_draft_probability = (
                draft_probability
                .gather(
                    1,
                    current_draft_token
                    .unsqueeze(1),
                )
                .squeeze(1)
            )

            acceptance_probability = (
                torch.minimum(
                    torch.ones_like(
                        selected_target_probability
                    ),
                    selected_target_probability
                    / selected_draft_probability
                    .clamp_min(epsilon),
                )
            )

            acceptance_uniform = torch.rand(
                current_draft_token.shape[0],
                device=device,
                generator=acceptance_generator,
            )

            accepted = bool(
                (
                    acceptance_uniform
                    < acceptance_probability
                ).item()
            )

        acceptance_probabilities.append(
            float(
                acceptance_probability.item()
            )
        )

        if accepted:
            accepted_count += 1
            continue

        rejected_position = step

        if temperature <= 0:
            correction_token = (
                target_logits.argmax(
                    dim=-1
                )
            )
        else:
            target_probability = F.softmax(
                target_logits / temperature,
                dim=-1,
            )

            residual_probability = (
                target_probability
                - draft_probability
            ).clamp_min(0.0)

            residual_mass = (
                residual_probability.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            # Numerical fallback for p approximately equal to q.
            use_target_fallback = (
                residual_mass.squeeze(1)
                <= epsilon
            )

            normalized_residual = (
                residual_probability
                / residual_mass.clamp_min(
                    epsilon
                )
            )

            if bool(
                use_target_fallback.item()
            ):
                normalized_residual = (
                    target_probability
                )

            correction_token = (
                sample_from_probability(
                    normalized_residual,
                    sampling_generator,
                )
            )

        break

    if rejected_position is not None:
        committed_draft_tokens = (
            draft_token_ids[
                :,
                :accepted_count,
            ]
        )

        rollback_length = (
            base_sequence_length
            + accepted_count
        )

        rolled_back_cache = crop_cache(
            verification_result["cache"],
            rollback_length,
        )

        correction_state = (
            advance_target_one_token(
                token_ids=correction_token,
                cache=rolled_back_cache,
                previous_sequence_length=
                    rollback_length,
            )
        )

        emitted_token_ids = torch.cat(
            [
                committed_draft_tokens,
                correction_token.view(1, 1),
            ],
            dim=1,
        )

        next_target_state = (
            correction_state
        )

        all_drafts_accepted = False

    else:
        bonus_logits = (
            verification_result[
                "logits"
            ][:, -1, :]
        )

        bonus_token = sample_from_logits(
            logits=bonus_logits,
            generator=sampling_generator,
            temperature=temperature,
        )

        bonus_state = advance_target_one_token(
            token_ids=bonus_token,
            cache=verification_result[
                "cache"
            ],
            previous_sequence_length=
                verification_result[
                    "sequence_length"
                ],
        )

        emitted_token_ids = torch.cat(
            [
                draft_token_ids,
                bonus_token.view(1, 1),
            ],
            dim=1,
        )

        next_target_state = bonus_state
        all_drafts_accepted = True

    return {
        "emitted_token_ids":
            emitted_token_ids,

        "next_target_state":
            next_target_state,

        "draft_token_ids":
            draft_token_ids,

        "accepted_count":
            accepted_count,

        "rejected_position":
            rejected_position,

        "all_drafts_accepted":
            all_drafts_accepted,

        "correction_token":
            correction_token,

        "bonus_token":
            bonus_token,

        "acceptance_probabilities":
            acceptance_probabilities,
    }

In [22]:
@torch.inference_mode()
def generate_vanilla_cached(
    prompt_token_ids,
    num_new_tokens,
    temperature=0.0,
    seed=42,
):
    prompt_token_ids = (
        prompt_token_ids.to(device)
    )

    generator = torch.Generator(
        device=device.type
    )

    generator.manual_seed(seed)

    state = prefill_target(
        prompt_token_ids
    )

    generated_tokens = []

    for _ in range(num_new_tokens):
        next_token = sample_from_logits(
            logits=state[
                "next_token_logits"
            ],
            generator=generator,
            temperature=temperature,
        )

        generated_tokens.append(
            next_token
        )

        state = advance_target_one_token(
            token_ids=next_token,
            cache=state["cache"],
            previous_sequence_length=
                state["sequence_length"],
        )

    return torch.stack(
        generated_tokens,
        dim=1,
    )

In [23]:
@torch.inference_mode()
def generate_speculative_cached(
    prompt_token_ids,
    draft_model,
    horizon,
    num_new_tokens,
    temperature=0.0,
    seed=42,
):
    prompt_token_ids = (
        prompt_token_ids.to(device)
    )

    sampling_generator = torch.Generator(
        device=device.type
    )

    sampling_generator.manual_seed(
        seed
    )

    acceptance_generator = torch.Generator(
        device=device.type
    )

    acceptance_generator.manual_seed(
        seed + 100000
    )

    state = prefill_target(
        prompt_token_ids
    )

    generated_token_chunks = []
    total_generated = 0
    round_records = []

    round_index = 0

    while total_generated < num_new_tokens:
        remaining_tokens = (
            num_new_tokens
            - total_generated
        )

        current_horizon = min(
            horizon,
            max(1, remaining_tokens),
        )

        round_result = (
            speculative_decode_round(
                draft_model=draft_model,
                target_state=state,
                horizon=current_horizon,
                sampling_generator=
                    sampling_generator,
                acceptance_generator=
                    acceptance_generator,
                temperature=temperature,
            )
        )

        emitted_tokens = round_result[
            "emitted_token_ids"
        ]

        generated_token_chunks.append(
            emitted_tokens
        )

        total_generated += (
            emitted_tokens.shape[1]
        )

        state = round_result[
            "next_target_state"
        ]

        round_records.append({
            "round": round_index,
            "horizon":
                current_horizon,
            "accepted_count":
                round_result[
                    "accepted_count"
                ],
            "num_emitted_tokens":
                emitted_tokens.shape[1],
            "all_drafts_accepted":
                round_result[
                    "all_drafts_accepted"
                ],
            "rejected_position":
                round_result[
                    "rejected_position"
                ],
        })

        round_index += 1

    generated_tokens = torch.cat(
        generated_token_chunks,
        dim=1,
    )

    generated_tokens = (
        generated_tokens[
            :,
            :num_new_tokens,
        ]
    )

    return {
        "generated_token_ids":
            generated_tokens,
        "round_records":
            pd.DataFrame(
                round_records
            ),
    }

In [24]:
CORRECTNESS_NEW_TOKENS = 32

correctness_prompt = (
    test_prompt_tokens[:1].to(device)
)

vanilla_greedy_tokens = (
    generate_vanilla_cached(
        prompt_token_ids=
            correctness_prompt,
        num_new_tokens=
            CORRECTNESS_NEW_TOKENS,
        temperature=0.0,
        seed=SEED,
    )
)

greedy_correctness_rows = []

for method_name, draft_model in (
    draft_models.items()
):
    for horizon in [2, 4, 8]:
        speculative_result = (
            generate_speculative_cached(
                prompt_token_ids=
                    correctness_prompt,
                draft_model=
                    draft_model,
                horizon=horizon,
                num_new_tokens=
                    CORRECTNESS_NEW_TOKENS,
                temperature=0.0,
                seed=SEED,
            )
        )

        speculative_tokens = (
            speculative_result[
                "generated_token_ids"
            ]
        )

        exact_match = bool(
            torch.equal(
                vanilla_greedy_tokens,
                speculative_tokens,
            )
        )

        first_difference = None

        if not exact_match:
            difference_positions = (
                vanilla_greedy_tokens
                != speculative_tokens
            ).nonzero(
                as_tuple=False
            )

            first_difference = int(
                difference_positions[
                    0,
                    1,
                ].item()
            )

        greedy_correctness_rows.append({
            "method": method_name,
            "horizon": horizon,
            "exact_match":
                exact_match,
            "first_difference":
                first_difference,
            "num_rounds":
                len(
                    speculative_result[
                        "round_records"
                    ]
                ),
            "mean_accepted_drafts":
                speculative_result[
                    "round_records"
                ][
                    "accepted_count"
                ].mean(),
        })

greedy_correctness_df = pd.DataFrame(
    greedy_correctness_rows
)

print(
    greedy_correctness_df.to_string(
        index=False
    )
)

assert greedy_correctness_df[
    "exact_match"
].all()

print(
    "All greedy speculative outputs "
    "exactly match vanilla decoding."
)

    method  horizon  exact_match first_difference  num_rounds  mean_accepted_drafts
Multi-step        2         True             None          12              1.666667
Multi-step        4         True             None           9              2.666667
Multi-step        8         True             None           8              3.125000
Decoder-KL        2         True             None          12              1.666667
Decoder-KL        4         True             None           9              2.555556
Decoder-KL        8         True             None           9              2.666667
  Local-LK        2         True             None          12              1.666667
  Local-LK        4         True             None          10              2.300000
  Local-LK        8         True             None          10              2.300000
All greedy speculative outputs exactly match vanilla decoding.


In [25]:
print("Prompt:")
print(
    tokenizer.decode(
        correctness_prompt[0].tolist()
    )
)

print("\nVanilla continuation:")
print(
    tokenizer.decode(
        vanilla_greedy_tokens[
            0
        ].tolist()
    )
)

example_speculative = (
    generate_speculative_cached(
        prompt_token_ids=
            correctness_prompt,
        draft_model=
            draft_models["Local-LK"],
        horizon=4,
        num_new_tokens=
            CORRECTNESS_NEW_TOKENS,
        temperature=0.0,
        seed=SEED,
    )
)

print("\nSpeculative continuation:")
print(
    tokenizer.decode(
        example_speculative[
            "generated_token_ids"
        ][0].tolist()
    )
)

print("\nRound diagnostics:")
print(
    example_speculative[
        "round_records"
    ].to_string(index=False)
)

Prompt:
 .<|endoftext|>His father died around 740 . Du Fu would have been allowed to enter the civil service because of his father 's rank , but he is thought

Vanilla continuation:
 to have died in the same way that his father died.





















Speculative continuation:
 to have died in the same way that his father died.





















Round diagnostics:
 round  horizon  accepted_count  num_emitted_tokens  all_drafts_accepted  rejected_position
     0        4               1                   2                False                1.0
     1        4               3                   4                False                3.0
     2        4               1                   2                False                1.0
     3        4               2                   3                False                2.0
     4        4               3                   4                False                3.0
     5        4               2                   3                False   

In [26]:
stochastic_result = (
    generate_speculative_cached(
        prompt_token_ids=
            correctness_prompt,
        draft_model=
            draft_models["Local-LK"],
        horizon=4,
        num_new_tokens=32,
        temperature=1.0,
        seed=SEED,
    )
)

stochastic_tokens = (
    stochastic_result[
        "generated_token_ids"
    ]
)

print(
    "Stochastic token shape:",
    stochastic_tokens.shape,
)

print("\nStochastic continuation:")
print(
    tokenizer.decode(
        stochastic_tokens[
            0
        ].tolist()
    )
)

print("\nStochastic rounds:")
print(
    stochastic_result[
        "round_records"
    ].to_string(index=False)
)

assert stochastic_tokens.shape == (
    1,
    32,
)

assert (
    stochastic_result[
        "round_records"
    ]["accepted_count"]
    >= 0
).all()

print("Stochastic smoke test passed.")

Stochastic token shape: torch.Size([1, 32])

Stochastic continuation:
 to have been a retired officer from Mestler County A New Jersey for years.




Some of the names on this page are incorrect.

Stochastic rounds:
 round  horizon  accepted_count  num_emitted_tokens  all_drafts_accepted  rejected_position
     0        4               1                   2                False                1.0
     1        4               1                   2                False                1.0
     2        4               1                   2                False                1.0
     3        4               2                   3                False                2.0
     4        4               1                   2                False                1.0
     5        4               2                   3                False                2.0
     6        4               1                   2                False                1.0
     7        4               2                 

In [27]:
print("=" * 100)
print("NOTEBOOK 20 — PHASE B SUMMARY")
print("=" * 100)

print("\nGreedy correctness:")

print(
    greedy_correctness_df.to_string(
        index=False
    )
)

print(
    "\nAll exact matches:",
    bool(
        greedy_correctness_df[
            "exact_match"
        ].all()
    ),
)

print(
    "\nStochastic generated shape:",
    tuple(stochastic_tokens.shape),
)

print(
    "Stochastic number of rounds:",
    len(
        stochastic_result[
            "round_records"
        ]
    ),
)

print("\nPHASE B COMPLETE")

NOTEBOOK 20 — PHASE B SUMMARY

Greedy correctness:
    method  horizon  exact_match first_difference  num_rounds  mean_accepted_drafts
Multi-step        2         True             None          12              1.666667
Multi-step        4         True             None           9              2.666667
Multi-step        8         True             None           8              3.125000
Decoder-KL        2         True             None          12              1.666667
Decoder-KL        4         True             None           9              2.555556
Decoder-KL        8         True             None           9              2.666667
  Local-LK        2         True             None          12              1.666667
  Local-LK        4         True             None          10              2.300000
  Local-LK        8         True             None          10              2.300000

All exact matches: True

Stochastic generated shape: (1, 32)
Stochastic number of rounds: 13

PHASE B COMPLE

In [28]:
@torch.inference_mode()
def benchmark_vanilla_once(
    prompt_token_ids,
    num_new_tokens,
    temperature,
    seed,
):
    generator = torch.Generator(
        device=device.type
    )

    generator.manual_seed(seed)

    synchronize_device()
    prefill_start = time.perf_counter()

    state = prefill_target(
        prompt_token_ids
    )

    synchronize_device()
    prefill_seconds = (
        time.perf_counter()
        - prefill_start
    )

    generated_tokens = []

    torch.cuda.reset_peak_memory_stats()

    synchronize_device()
    decode_start = time.perf_counter()

    for _ in range(num_new_tokens):
        next_token = sample_from_logits(
            logits=state[
                "next_token_logits"
            ],
            generator=generator,
            temperature=temperature,
        )

        generated_tokens.append(
            next_token
        )

        state = advance_target_one_token(
            token_ids=next_token,
            cache=state["cache"],
            previous_sequence_length=
                state["sequence_length"],
        )

    synchronize_device()

    decode_seconds = (
        time.perf_counter()
        - decode_start
    )

    peak_memory_mb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    generated_token_ids = torch.stack(
        generated_tokens,
        dim=1,
    )

    return {
        "generated_token_ids":
            generated_token_ids,
        "prefill_seconds":
            prefill_seconds,
        "decode_seconds":
            decode_seconds,
        "tokens_per_second":
            num_new_tokens
            / decode_seconds,
        "num_target_decode_calls":
            num_new_tokens,
        "peak_memory_mb":
            peak_memory_mb,
    }

In [29]:
@torch.inference_mode()
def benchmark_speculative_once(
    prompt_token_ids,
    draft_model,
    horizon,
    num_new_tokens,
    temperature,
    seed,
):
    sampling_generator = torch.Generator(
        device=device.type
    )

    sampling_generator.manual_seed(seed)

    acceptance_generator = torch.Generator(
        device=device.type
    )

    acceptance_generator.manual_seed(
        seed + 100000
    )

    synchronize_device()
    prefill_start = time.perf_counter()

    state = prefill_target(
        prompt_token_ids
    )

    synchronize_device()
    prefill_seconds = (
        time.perf_counter()
        - prefill_start
    )

    generated_chunks = []
    generated_count = 0
    round_rows = []

    torch.cuda.reset_peak_memory_stats()

    synchronize_device()
    decode_start = time.perf_counter()

    while generated_count < num_new_tokens:
        remaining = (
            num_new_tokens
            - generated_count
        )

        current_horizon = min(
            horizon,
            max(1, remaining),
        )

        round_result = (
            speculative_decode_round(
                draft_model=draft_model,
                target_state=state,
                horizon=current_horizon,
                sampling_generator=
                    sampling_generator,
                acceptance_generator=
                    acceptance_generator,
                temperature=temperature,
            )
        )

        emitted_tokens = round_result[
            "emitted_token_ids"
        ]

        generated_chunks.append(
            emitted_tokens
        )

        emitted_count = (
            emitted_tokens.shape[1]
        )

        generated_count += emitted_count

        state = round_result[
            "next_target_state"
        ]

        round_rows.append({
            "horizon":
                current_horizon,

            "accepted_count":
                round_result[
                    "accepted_count"
                ],

            "emitted_count":
                emitted_count,

            "all_drafts_accepted":
                round_result[
                    "all_drafts_accepted"
                ],
        })

    synchronize_device()

    decode_seconds = (
        time.perf_counter()
        - decode_start
    )

    peak_memory_mb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    generated_token_ids = torch.cat(
        generated_chunks,
        dim=1,
    )[:, :num_new_tokens]

    round_df = pd.DataFrame(
        round_rows
    )

    num_rounds = len(round_df)

    # Each round performs:
    # 1 target block verification
    # 1 correction/bonus target update
    num_target_decode_calls = (
        2 * num_rounds
    )

    return {
        "generated_token_ids":
            generated_token_ids,

        "round_records":
            round_df,

        "prefill_seconds":
            prefill_seconds,

        "decode_seconds":
            decode_seconds,

        "tokens_per_second":
            num_new_tokens
            / decode_seconds,

        "num_rounds":
            num_rounds,

        "num_target_decode_calls":
            num_target_decode_calls,

        "target_calls_per_output_token":
            num_target_decode_calls
            / num_new_tokens,

        "mean_accepted_drafts":
            round_df[
                "accepted_count"
            ].mean(),

        "mean_emitted_tokens":
            round_df[
                "emitted_count"
            ].mean(),

        "all_acceptance_rate":
            round_df[
                "all_drafts_accepted"
            ].mean(),

        "peak_memory_mb":
            peak_memory_mb,
    }

In [30]:
BENCHMARK_NUM_PROMPTS = 10
BENCHMARK_NEW_TOKENS = 64
BENCHMARK_REPETITIONS = 3

BENCHMARK_TEMPERATURES = {
    "greedy": 0.0,
    "stochastic_t1": 1.0,
}

BENCHMARK_HORIZONS = [
    2,
    4,
    8,
]

benchmark_prompts = (
    test_prompt_tokens[
        :BENCHMARK_NUM_PROMPTS
    ]
)

print(
    "Benchmark prompts:",
    benchmark_prompts.shape,
)

print(
    "New tokens per generation:",
    BENCHMARK_NEW_TOKENS,
)

print(
    "Repetitions:",
    BENCHMARK_REPETITIONS,
)

Benchmark prompts: torch.Size([10, 32])
New tokens per generation: 64
Repetitions: 3


In [31]:
print("Warming up GPU...")

warmup_prompt = (
    benchmark_prompts[:1].to(device)
)

for warmup_index in range(3):
    _ = benchmark_vanilla_once(
        prompt_token_ids=
            warmup_prompt,
        num_new_tokens=16,
        temperature=0.0,
        seed=SEED + warmup_index,
    )

    _ = benchmark_speculative_once(
        prompt_token_ids=
            warmup_prompt,
        draft_model=
            draft_models["Multi-step"],
        horizon=4,
        num_new_tokens=16,
        temperature=0.0,
        seed=SEED + warmup_index,
    )

gc.collect()
torch.cuda.empty_cache()
synchronize_device()

print("Warm-up complete.")

Warming up GPU...
Warm-up complete.


In [32]:
benchmark_rows = []

for temperature_name, temperature in (
    BENCHMARK_TEMPERATURES.items()
):
    for prompt_index in range(
        BENCHMARK_NUM_PROMPTS
    ):
        prompt = benchmark_prompts[
            prompt_index:
            prompt_index + 1
        ].to(device)

        for repetition in range(
            BENCHMARK_REPETITIONS
        ):
            run_seed = (
                SEED
                + prompt_index * 100
                + repetition
            )

            result = benchmark_vanilla_once(
                prompt_token_ids=prompt,
                num_new_tokens=
                    BENCHMARK_NEW_TOKENS,
                temperature=temperature,
                seed=run_seed,
            )

            benchmark_rows.append({
                "method": "Vanilla AR",
                "horizon": 1,
                "temperature_name":
                    temperature_name,
                "temperature":
                    temperature,
                "prompt_index":
                    prompt_index,
                "repetition":
                    repetition,
                "decode_seconds":
                    result[
                        "decode_seconds"
                    ],
                "tokens_per_second":
                    result[
                        "tokens_per_second"
                    ],
                "num_target_decode_calls":
                    result[
                        "num_target_decode_calls"
                    ],
                "target_calls_per_output_token":
                    1.0,
                "mean_accepted_drafts":
                    np.nan,
                "mean_emitted_tokens":
                    1.0,
                "all_acceptance_rate":
                    np.nan,
                "peak_memory_mb":
                    result[
                        "peak_memory_mb"
                    ],
            })

        print(
            f"Vanilla {temperature_name}: "
            f"prompt {prompt_index + 1}/"
            f"{BENCHMARK_NUM_PROMPTS}"
        )

Vanilla greedy: prompt 1/10
Vanilla greedy: prompt 2/10
Vanilla greedy: prompt 3/10
Vanilla greedy: prompt 4/10
Vanilla greedy: prompt 5/10
Vanilla greedy: prompt 6/10
Vanilla greedy: prompt 7/10
Vanilla greedy: prompt 8/10
Vanilla greedy: prompt 9/10
Vanilla greedy: prompt 10/10
Vanilla stochastic_t1: prompt 1/10
Vanilla stochastic_t1: prompt 2/10
Vanilla stochastic_t1: prompt 3/10
Vanilla stochastic_t1: prompt 4/10
Vanilla stochastic_t1: prompt 5/10
Vanilla stochastic_t1: prompt 6/10
Vanilla stochastic_t1: prompt 7/10
Vanilla stochastic_t1: prompt 8/10
Vanilla stochastic_t1: prompt 9/10
Vanilla stochastic_t1: prompt 10/10


In [33]:
for method_name, draft_model in (
    draft_models.items()
):
    for horizon in BENCHMARK_HORIZONS:
        for temperature_name, temperature in (
            BENCHMARK_TEMPERATURES.items()
        ):
            for prompt_index in range(
                BENCHMARK_NUM_PROMPTS
            ):
                prompt = benchmark_prompts[
                    prompt_index:
                    prompt_index + 1
                ].to(device)

                for repetition in range(
                    BENCHMARK_REPETITIONS
                ):
                    run_seed = (
                        SEED
                        + prompt_index * 100
                        + repetition
                    )

                    result = (
                        benchmark_speculative_once(
                            prompt_token_ids=
                                prompt,
                            draft_model=
                                draft_model,
                            horizon=horizon,
                            num_new_tokens=
                                BENCHMARK_NEW_TOKENS,
                            temperature=
                                temperature,
                            seed=run_seed,
                        )
                    )

                    benchmark_rows.append({
                        "method":
                            method_name,

                        "horizon":
                            horizon,

                        "temperature_name":
                            temperature_name,

                        "temperature":
                            temperature,

                        "prompt_index":
                            prompt_index,

                        "repetition":
                            repetition,

                        "decode_seconds":
                            result[
                                "decode_seconds"
                            ],

                        "tokens_per_second":
                            result[
                                "tokens_per_second"
                            ],

                        "num_target_decode_calls":
                            result[
                                "num_target_decode_calls"
                            ],

                        "target_calls_per_output_token":
                            result[
                                "target_calls_per_output_token"
                            ],

                        "mean_accepted_drafts":
                            result[
                                "mean_accepted_drafts"
                            ],

                        "mean_emitted_tokens":
                            result[
                                "mean_emitted_tokens"
                            ],

                        "all_acceptance_rate":
                            result[
                                "all_acceptance_rate"
                            ],

                        "peak_memory_mb":
                            result[
                                "peak_memory_mb"
                            ],
                    })

                print(
                    f"{method_name}, "
                    f"gamma={horizon}, "
                    f"{temperature_name}: "
                    f"prompt {prompt_index + 1}/"
                    f"{BENCHMARK_NUM_PROMPTS}"
                )

benchmark_df = pd.DataFrame(
    benchmark_rows
)

print(
    "Benchmark rows:",
    len(benchmark_df),
)

Multi-step, gamma=2, greedy: prompt 1/10
Multi-step, gamma=2, greedy: prompt 2/10
Multi-step, gamma=2, greedy: prompt 3/10
Multi-step, gamma=2, greedy: prompt 4/10
Multi-step, gamma=2, greedy: prompt 5/10
Multi-step, gamma=2, greedy: prompt 6/10
Multi-step, gamma=2, greedy: prompt 7/10
Multi-step, gamma=2, greedy: prompt 8/10
Multi-step, gamma=2, greedy: prompt 9/10
Multi-step, gamma=2, greedy: prompt 10/10
Multi-step, gamma=2, stochastic_t1: prompt 1/10
Multi-step, gamma=2, stochastic_t1: prompt 2/10
Multi-step, gamma=2, stochastic_t1: prompt 3/10
Multi-step, gamma=2, stochastic_t1: prompt 4/10
Multi-step, gamma=2, stochastic_t1: prompt 5/10
Multi-step, gamma=2, stochastic_t1: prompt 6/10
Multi-step, gamma=2, stochastic_t1: prompt 7/10
Multi-step, gamma=2, stochastic_t1: prompt 8/10
Multi-step, gamma=2, stochastic_t1: prompt 9/10
Multi-step, gamma=2, stochastic_t1: prompt 10/10
Multi-step, gamma=4, greedy: prompt 1/10
Multi-step, gamma=4, greedy: prompt 2/10
Multi-step, gamma=4, greed

In [34]:
vanilla_reference = (
    benchmark_df[
        benchmark_df["method"]
        == "Vanilla AR"
    ][
        [
            "temperature_name",
            "prompt_index",
            "repetition",
            "decode_seconds",
            "tokens_per_second",
        ]
    ]
    .rename(columns={
        "decode_seconds":
            "vanilla_decode_seconds",

        "tokens_per_second":
            "vanilla_tokens_per_second",
    })
)

benchmark_with_speedup = (
    benchmark_df.merge(
        vanilla_reference,
        on=[
            "temperature_name",
            "prompt_index",
            "repetition",
        ],
        how="left",
    )
)

benchmark_with_speedup[
    "speedup_over_vanilla"
] = (
    benchmark_with_speedup[
        "vanilla_decode_seconds"
    ]
    / benchmark_with_speedup[
        "decode_seconds"
    ]
)

assert benchmark_with_speedup[
    "vanilla_decode_seconds"
].notna().all()

In [35]:
benchmark_summary = (
    benchmark_with_speedup
    .groupby(
        [
            "method",
            "horizon",
            "temperature_name",
        ],
        dropna=False,
    )
    .agg(
        num_runs=(
            "decode_seconds",
            "size",
        ),

        median_decode_seconds=(
            "decode_seconds",
            "median",
        ),

        mean_tokens_per_second=(
            "tokens_per_second",
            "mean",
        ),

        median_tokens_per_second=(
            "tokens_per_second",
            "median",
        ),

        mean_speedup=(
            "speedup_over_vanilla",
            "mean",
        ),

        median_speedup=(
            "speedup_over_vanilla",
            "median",
        ),

        speedup_std=(
            "speedup_over_vanilla",
            "std",
        ),

        mean_target_calls_per_token=(
            "target_calls_per_output_token",
            "mean",
        ),

        mean_accepted_drafts=(
            "mean_accepted_drafts",
            "mean",
        ),

        mean_emitted_tokens=(
            "mean_emitted_tokens",
            "mean",
        ),

        mean_all_acceptance_rate=(
            "all_acceptance_rate",
            "mean",
        ),

        mean_peak_memory_mb=(
            "peak_memory_mb",
            "mean",
        ),
    )
    .reset_index()
)

print(
    benchmark_summary
    .sort_values(
        [
            "temperature_name",
            "method",
            "horizon",
        ]
    )
    .to_string(index=False)
)

    method  horizon temperature_name  num_runs  median_decode_seconds  mean_tokens_per_second  median_tokens_per_second  mean_speedup  median_speedup  speedup_std  mean_target_calls_per_token  mean_accepted_drafts  mean_emitted_tokens  mean_all_acceptance_rate  mean_peak_memory_mb
Decoder-KL        2           greedy        30               0.118913              537.062158                538.264725      0.597443        0.598513     0.082048                     0.846875              1.425953             2.425953                  0.437327           376.462386
Decoder-KL        4           greedy        30               0.119669              586.295986                534.815102      0.652445        0.594460     0.174488                     0.778125              1.774397             2.774397                  0.143839           377.619173
Decoder-KL        8           greedy        30               0.126442              580.619533                506.162953      0.646284        0.561783     

In [36]:
def bootstrap_mean_speedup_ci(
    speedup_values,
    num_bootstrap=10000,
    seed=42,
):
    values = np.asarray(
        speedup_values,
        dtype=np.float64,
    )

    rng = np.random.default_rng(seed)

    bootstrap_means = np.empty(
        num_bootstrap
    )

    for index in range(num_bootstrap):
        sampled = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        bootstrap_means[index] = (
            sampled.mean()
        )

    return {
        "mean_speedup":
            values.mean(),

        "ci_lower":
            np.percentile(
                bootstrap_means,
                2.5,
            ),

        "ci_upper":
            np.percentile(
                bootstrap_means,
                97.5,
            ),

        "probability_speedup_gt_1":
            np.mean(
                bootstrap_means > 1.0
            ),
    }

In [37]:
speedup_ci_rows = []

speculative_benchmark = (
    benchmark_with_speedup[
        benchmark_with_speedup[
            "method"
        ] != "Vanilla AR"
    ]
)

for group_keys, group_df in (
    speculative_benchmark.groupby(
        [
            "method",
            "horizon",
            "temperature_name",
        ]
    )
):
    (
        method_name,
        horizon,
        temperature_name,
    ) = group_keys

    ci_result = (
        bootstrap_mean_speedup_ci(
            group_df[
                "speedup_over_vanilla"
            ].to_numpy(),
            num_bootstrap=10000,
            seed=SEED + horizon,
        )
    )

    speedup_ci_rows.append({
        "method": method_name,
        "horizon": horizon,
        "temperature_name":
            temperature_name,
        **ci_result,
    })

speedup_ci_df = pd.DataFrame(
    speedup_ci_rows
)

print(
    speedup_ci_df
    .sort_values(
        [
            "temperature_name",
            "method",
            "horizon",
        ]
    )
    .to_string(index=False)
)

    method  horizon temperature_name  mean_speedup  ci_lower  ci_upper  probability_speedup_gt_1
Decoder-KL        2           greedy      0.597443  0.568450  0.627118                       0.0
Decoder-KL        4           greedy      0.652445  0.592679  0.715132                       0.0
Decoder-KL        8           greedy      0.646284  0.561347  0.734270                       0.0
  Local-LK        2           greedy      0.594609  0.565988  0.623806                       0.0
  Local-LK        4           greedy      0.637154  0.584849  0.691704                       0.0
  Local-LK        8           greedy      0.621289  0.561595  0.684139                       0.0
Multi-step        2           greedy      0.597943  0.569360  0.627553                       0.0
Multi-step        4           greedy      0.676020  0.607522  0.748415                       0.0
Multi-step        8           greedy      0.719680  0.611435  0.837114                       0.0
Decoder-KL        2    stochas

In [38]:
benchmark_with_speedup.to_csv(
    OUTPUT_TABLE_DIR
    / "20_end_to_end_benchmark_runs.csv",
    index=False,
)

benchmark_summary.to_csv(
    OUTPUT_TABLE_DIR
    / "20_end_to_end_benchmark_summary.csv",
    index=False,
)

speedup_ci_df.to_csv(
    OUTPUT_TABLE_DIR
    / "20_end_to_end_speedup_bootstrap.csv",
    index=False,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 280)

print("=" * 110)
print("TABLE 1: END-TO-END BENCHMARK SUMMARY")
print("=" * 110)

print(
    benchmark_summary
    .sort_values(
        [
            "temperature_name",
            "method",
            "horizon",
        ]
    )
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("TABLE 2: SPEEDUP BOOTSTRAP")
print("=" * 110)

print(
    speedup_ci_df
    .sort_values(
        [
            "temperature_name",
            "method",
            "horizon",
        ]
    )
    .to_string(index=False)
)

print("\nPHASE C COMPLETE")

TABLE 1: END-TO-END BENCHMARK SUMMARY
    method  horizon temperature_name  num_runs  median_decode_seconds  mean_tokens_per_second  median_tokens_per_second  mean_speedup  median_speedup  speedup_std  mean_target_calls_per_token  mean_accepted_drafts  mean_emitted_tokens  mean_all_acceptance_rate  mean_peak_memory_mb
Decoder-KL        2           greedy        30               0.118913              537.062158                538.264725      0.597443        0.598513     0.082048                     0.846875              1.425953             2.425953                  0.437327           376.462386
Decoder-KL        4           greedy        30               0.119669              586.295986                534.815102      0.652445        0.594460     0.174488                     0.778125              1.774397             2.774397                  0.143839           377.619173
Decoder-KL        8           greedy        30               0.126442              580.619533                506.162

In [39]:
PROFILE_REPETITIONS = 100

profile_prompt = (
    test_prompt_tokens[:1].to(device)
)

profile_base_state = prefill_target(
    profile_prompt
)

profile_rows = []

for method_name, draft_model in (
    draft_models.items()
):
    for horizon in [2, 4, 8]:
        draft_times = []
        verification_times = []
        update_times = []

        for repetition in range(
            PROFILE_REPETITIONS
        ):
            generator = torch.Generator(
                device=device.type
            )

            generator.manual_seed(
                SEED + repetition
            )

            with CudaTimer() as draft_timer:
                draft_result = (
                    generate_latent_draft_block(
                        draft_model=draft_model,
                        initial_hidden=
                            profile_base_state[
                                "last_hidden"
                            ],
                        horizon=horizon,
                        generator=generator,
                        temperature=1.0,
                    )
                )

            draft_times.append(
                draft_timer.elapsed_seconds
            )

            verification_cache = copy.deepcopy(
                profile_base_state["cache"]
            )

            with CudaTimer() as verification_timer:
                verification_result = (
                    verify_target_block(
                        draft_token_ids=
                            draft_result[
                                "draft_token_ids"
                            ],
                        cache=
                            verification_cache,
                        previous_sequence_length=
                            profile_base_state[
                                "sequence_length"
                            ],
                    )
                )

            verification_times.append(
                verification_timer.elapsed_seconds
            )

            bonus_token = (
                verification_result[
                    "logits"
                ][:, -1, :]
                .argmax(dim=-1)
            )

            with CudaTimer() as update_timer:
                _ = advance_target_one_token(
                    token_ids=bonus_token,
                    cache=verification_result[
                        "cache"
                    ],
                    previous_sequence_length=
                        verification_result[
                            "sequence_length"
                        ],
                )

            update_times.append(
                update_timer.elapsed_seconds
            )

        profile_rows.append({
            "method": method_name,
            "horizon": horizon,

            "draft_latency_ms":
                1000
                * np.median(
                    draft_times
                ),

            "verification_latency_ms":
                1000
                * np.median(
                    verification_times
                ),

            "update_latency_ms":
                1000
                * np.median(
                    update_times
                ),

            "estimated_round_latency_ms":
                1000
                * (
                    np.median(draft_times)
                    + np.median(
                        verification_times
                    )
                    + np.median(update_times)
                ),
        })

component_profile_df = pd.DataFrame(
    profile_rows
)

print(
    component_profile_df.to_string(
        index=False
    )
)

    method  horizon  draft_latency_ms  verification_latency_ms  update_latency_ms  estimated_round_latency_ms
Multi-step        2          0.440490                 3.119875           1.258845                    4.819210
Multi-step        4          0.867125                 3.166245           1.260175                    5.293545
Multi-step        8          1.720710                 3.143375           1.269855                    6.133940
Decoder-KL        2          0.440285                 3.126865           1.259455                    4.826605
Decoder-KL        4          0.866870                 3.157130           1.258335                    5.282335
Decoder-KL        8          1.721095                 3.153940           1.270585                    6.145620
  Local-LK        2          0.439950                 3.133345           1.261245                    4.834540
  Local-LK        4          0.867405                 3.198550           1.268430                    5.334385
  Local-LK

In [40]:
vanilla_update_times = []

for repetition in range(
    PROFILE_REPETITIONS
):
    fresh_state = prefill_target(
        profile_prompt
    )

    next_token = (
        fresh_state[
            "next_token_logits"
        ].argmax(dim=-1)
    )

    with CudaTimer() as timer:
        _ = advance_target_one_token(
            token_ids=next_token,
            cache=fresh_state["cache"],
            previous_sequence_length=
                fresh_state[
                    "sequence_length"
                ],
        )

    vanilla_update_times.append(
        timer.elapsed_seconds
    )

vanilla_update_latency_ms = (
    1000
    * np.median(
        vanilla_update_times
    )
)

print(
    "Vanilla one-token update latency (ms):",
    vanilla_update_latency_ms,
)

Vanilla one-token update latency (ms): 1.1855099996864737


In [41]:
component_profile_df.to_csv(
    OUTPUT_TABLE_DIR
    / "20_component_latency_profile.csv",
    index=False,
)

with open(
    OUTPUT_TABLE_DIR
    / "20_vanilla_update_latency.json",
    "w",
) as file:
    json.dump(
        {
            "vanilla_update_latency_ms":
                vanilla_update_latency_ms,
            "repetitions":
                PROFILE_REPETITIONS,
        },
        file,
        indent=2,
    )